# Model V
### Audio Spectrogram Transformer - trening sieci

Biblioteki

In [14]:
import sys
from pathlib import Path
import time

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

sys.path.append(str(Path.cwd().parent))

from model import InstrumentAST
from trening import get_device, train_model
from przygotowanie_danych import INSTRUMENTS, create_datasets, create_loaders


### Konfiguracja eksperymentu

In [15]:
# Identyfikator przygotowanego zbioru danych.
dataset = "1_1"
model_name = "AST"

epochs = 100
patience = 10
batch_size = 8
learning_rate = 0.0001

dropout = 0.1
threshold = 0.8

hidden_size = 384
num_hidden_layers = 6
num_attention_heads = 6
intermediate_size = 1536

results_file = Path("wyniki/modele.csv")


### Sprawdzenie, czy eksperyment był już wykonywany

In [16]:
if results_file.exists():
    results = pd.read_csv(results_file)

    combination_exists = (
        (results["Dataset"] == dataset) &
        (results["Model"] == model_name) &
        (results["Batch size"] == batch_size) &
        (results["Learning rate"] == learning_rate) &
        (results["Dropout"] == dropout) &
        (results["Hidden size"] == hidden_size) &
        (results["Transformer layers"] == num_hidden_layers) &
        (results["Attention heads"] == num_attention_heads) &
        (results["Intermediate size"] == intermediate_size)
    ).any()

    if combination_exists:
        raise ValueError(
            "Ta kombinacja datasetu i hiperparametrów "
            "była już testowana. Zatrzymuję program."
        )

print("Nowa kombinacja. Kontynuuję wykonywanie kodu.")

Nowa kombinacja. Kontynuuję wykonywanie kodu.


### Nadanie identyfikatora eksperymentu

In [17]:
results_dir = Path("wyniki")
results_dir.mkdir(parents=True, exist_ok=True)

existing_numbers = [
    int(f.stem)
    for f in results_dir.glob("*.pth")
    if f.stem.isdigit()
]

learning_id = max(existing_numbers, default=0) + 1

weights_filename = f"{learning_id}.pth"

print(f"Learning ID: {learning_id}")
print(f"Plik wag: {weights_filename}")

Learning ID: 1
Plik wag: 1.pth


### Przygotowanie danych

In [18]:
train_dataset, val_dataset, test_dataset = create_datasets(dataset)

train_loader, val_loader, test_loader = create_loaders(train_dataset, val_dataset, test_dataset, batch_size=batch_size)

print(f"Train: {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

# Pobranie przykładowego spektrogramu ze zbioru treningowego.
sample_X, sample_y = train_dataset[0]

# Automatyczne określenie wymiarów wejściowych AST.
num_mel_bins = sample_X.shape[-2]
max_length = sample_X.shape[-1]

print(f"Liczba pasm mel: {num_mel_bins}")
print(f"Długość sekwencji: {max_length}")

Wczytuję spektrogramy z: /Users/khaczkiewicz/Documents/projekt-instrumenty/instrument-recognition-project/spectrograms/1_1
Train: 7758
Validation: 1658
Test: 1664
Liczba pasm mel: 128
Długość sekwencji: 346


### Inicjalizacja modelu

In [19]:
device = get_device()
print(f"Urządzenie: {device}")

model = InstrumentAST(
    num_mel_bins=num_mel_bins,
    max_length=max_length,
    dropout=dropout
).to(device)

criterion = nn.MultiLabelSoftMarginLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Urządzenie: mps


### Trening sieci

In [20]:
start_time = time.perf_counter()

model, history, best_epoch, best_val_loss = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    patience=patience,
    model_name=weights_filename
)

training_time = time.perf_counter() - start_time

print(f"Czas treningu: {training_time / 60:.2f} min")

/opt/anaconda3/envs/cnn/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoka 1/100 | Train loss: 0.5686 | Val loss: 0.4989
Epoka 2/100 | Train loss: 0.4769 | Val loss: 0.4616
Epoka 3/100 | Train loss: 0.4344 | Val loss: 0.4045
Epoka 4/100 | Train loss: 0.4009 | Val loss: 0.3913
Epoka 5/100 | Train loss: 0.3768 | Val loss: 0.3778
Epoka 6/100 | Train loss: 0.3560 | Val loss: 0.3479
Epoka 7/100 | Train loss: 0.3345 | Val loss: 0.3570
Epoka 8/100 | Train loss: 0.3149 | Val loss: 0.3322
Epoka 9/100 | Train loss: 0.2967 | Val loss: 0.3177
Epoka 10/100 | Train loss: 0.2829 | Val loss: 0.3108
Epoka 11/100 | Train loss: 0.2649 | Val loss: 0.2953
Epoka 12/100 | Train loss: 0.2486 | Val loss: 0.3021
Epoka 13/100 | Train loss: 0.2337 | Val loss: 0.2842
Epoka 14/100 | Train loss: 0.2176 | Val loss: 0.2871
Epoka 15/100 | Train loss: 0.2022 | Val loss: 0.2731
Epoka 16/100 | Train loss: 0.1895 | Val loss: 0.2692
Epoka 17/100 | Train loss: 0.1769 | Val loss: 0.3016
Epoka 18/100 | Train loss: 0.1656 | Val loss: 0.2859
Epoka 19/100 | Train loss: 0.1515 | Val loss: 0.2984
Ep

### Zapis informacji o eksperymencie

In [21]:
experiment = pd.DataFrame([{
    "Learning_ID": learning_id,
    "Dataset": dataset,
    "Model": model_name,
    "Batch size": batch_size,
    "Learning rate": learning_rate,
    "Dropout": dropout,
    "Hidden size": hidden_size,
    "Transformer layers": num_hidden_layers,
    "Attention heads": num_attention_heads,
    "Intermediate size": intermediate_size,
    "Threshold": threshold,
    "Best epoch": best_epoch,
    "Best val loss": best_val_loss,
    "Training time [min]": training_time / 60
}])

experiment.to_csv(
    results_file,
    mode="a",
    header=not results_file.exists(),
    index=False
)

history_df = pd.DataFrame(history)

history_df.to_csv(
    results_dir / f"history_{learning_id}.csv",
    index_label="epoch"
)

print(f"Informacje o eksperymencie zapisano w: {results_file}")
print(f"Historia uczenia: history_{learning_id}.csv")

Informacje o eksperymencie zapisano w: wyniki/modele.csv
Historia uczenia: history_1.csv
